In [1]:
!pip install pandas mlxtend scikit-learn tensorflow

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 1.4/1.4 MB 17.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/331.9 MB ? eta -:--:--
    --------------------------------------- 6.6/331.9 MB 33.5 MB/s eta 0:00:10
   - -------------------------------------- 8.9/331.9 MB 22.1 MB/s eta 0:00:15
   -- ------------------------------------- 17.0/331.9 MB 26.8 MB/s eta 0:00:12
   --- ------------------------------------ 26.0/331.9 MB 31.0 MB/s eta 0:00:10
   --- ------------------------------------ 31.7/331.9 MB 30.5 MB/s eta 0:00:10
   ---- ----------------------------------- 34.1/331.9 MB 27.4 MB/s eta 0:00:11
   ---- ----------------------------------- 37.0/331.9 MB 25.2 MB/s eta 0:00:12
   ---- ----------------------------------- 39.6/331.9 MB 23.5 MB/s eta 0:00:13
   ----- ---------------------------------- 41.9/331.9 MB 22.4 MB/s eta 0:00:13
   ----- ---------------------------------- 47.7/331.9 MB 22.8 MB

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 6.33.0 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import re
import os

# Keras / TensorFlow imports for the Neural Network
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import skipgrams
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Dot, Reshape

# Helper for finding similar items
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully.")


Libraries imported successfully.


In [5]:
DATA_FILE = "OnlineRetail.csv" # Updated to use the full dataset
OUTPUT_FILE = "ergo_facts_neural_network.txt"

if not os.path.exists(DATA_FILE):
    print(f"Error: {DATA_FILE} not found.")
    print(f"Please make sure '{DATA_FILE}' (from the zip) is in the same directory as this notebook.")
else:
    print(f"Loading data from {DATA_FILE}...")
    try:
        df = pd.read_csv(DATA_FILE, encoding='ISO-8859-1')
        print("Data loaded successfully.")
    except Exception as e:
        print(f"Error loading file: {e}")


Loading data from OnlineRetail.csv...
Data loaded successfully.


In [9]:
print(f"Original row count: {len(df)}")

# 1. Remove rows with missing Description
df.dropna(subset=['Description'], inplace=True)

# 2. Convert InvoiceNo to string
df['InvoiceNo'] = df['InvoiceNo'].astype(str)

# 3. Remove canceled orders (InvoiceNo starts with 'C')
df = df[~df['InvoiceNo'].str.startswith('C')].copy()

# 4. Clean up Description text
df['Description'] = df['Description'].str.strip().str.lower()

Original row count: 541909


In [15]:
# Get a list of the 50 most common item descriptions
top_50_items = df['Description'].value_counts().head(50)
print(top_50_items)

Description
white hanging heart t-light holder     2327
jumbo bag red retrospot                2115
regency cakestand 3 tier               2019
party bunting                          1707
lunch bag red retrospot                1594
assorted colour bird ornament          1489
set of 3 cake tins pantry design       1399
pack of 72 retrospot cake cases        1370
lunch bag  black skull.                1328
natural slate heart chalkboard         1263
jumbo bag pink polkadot                1238
heart of wicker small                  1226
paper chain kit 50's christmas         1200
jumbo storage bag suki                 1197
jumbo shopper vintage red paisley      1190
lunch bag spaceboy design              1179
lunch bag cars blue                    1174
jam making set printed                 1169
spotty bunting                         1160
jam making set with jars               1142
recipe box pantry yellow design        1133
postage                                1126
lunch bag suki desig

In [19]:
print(f"Original row count: {len(df)}")

# 1. Remove rows with missing Description
df.dropna(subset=['Description'], inplace=True)

# 2. Convert InvoiceNo and StockCode to string
df['InvoiceNo'] = df['InvoiceNo'].astype(str)
df['StockCode'] = df['StockCode'].astype(str)

# 3. Remove canceled orders (InvoiceNo starts with 'C')
df = df[~df['InvoiceNo'].str.startswith('C')].copy()

# 4. Clean up Description text (do this *before* filtering)
df['Description'] = df['Description'].str.strip().str.lower()

# 5. Remove known non-product *descriptions*
# (Based on your EDA, 'postage' is the main one to remove)
non_products = ['postage', 'manual', 'dotcom postage', 'carrage', 'gift voucher', 'amazon fee', 'bank charges']
df = df[~df['Description'].isin(non_products)]

# 6. ADVANCED CLEANING: Remove items with non-standard StockCodes
# A real product code almost always has a number in it (e.g., '85123A'). 
# Non-products often have text codes (like 'POST' or 'MANUAL').
# This line keeps only the rows where the StockCode contains at least one digit.
df = df[df['StockCode'].str.contains(r'[0-9]')].copy()

print(f"Final cleaned row count: {len(df)}")

Original row count: 531167
Final cleaned row count: 528958


In [21]:
print("Grouping items by transaction (basket)...")

# Group by 'InvoiceNo' and aggregate all 'Description' into a list
transactions_list = df.groupby('InvoiceNo')['Description'].apply(list).values.tolist()

# Remove transactions with only one item, as they provide no context
transactions_list = [t for t in transactions_list if len(t) > 1]

print(f"Converted into {len(transactions_list)} valid transactions.")
print("\nExample Baskets:")
print(transactions_list[0])
print(transactions_list[1])

Grouping items by transaction (basket)...
Converted into 18302 valid transactions.

Example Baskets:
['white hanging heart t-light holder', 'white metal lantern', 'cream cupid hearts coat hanger', 'knitted union flag hot water bottle', 'red woolly hottie white heart.', 'set 7 babushka nesting boxes', 'glass star frosted t-light holder']
['hand warmer union jack', 'hand warmer red polka dot']


In [23]:
print("Tokenizing products...")
tokenizer = Tokenizer()
tokenizer.fit_on_texts(transactions_list)

# This is our product-to-integer mapping
product_index = tokenizer.word_index 
# We add 1 because index 0 is reserved
vocab_size = len(product_index) + 1

print(f"Vocabulary size (total unique products): {vocab_size}")
print("\nExample of product_index mapping:")
print(f"'hand warmer union jack' -> {product_index.get('hand warmer union jack')}")
print(f"'hand warmer red polka dot' -> {product_index.get('hand warmer red polka dot')}")

Tokenizing products...
Vocabulary size (total unique products): 4000

Example of product_index mapping:
'hand warmer union jack' -> 187
'hand warmer red polka dot' -> 2892


In [25]:
# [["hand warmer union jack"], ["alarm clock..."]] -> [[5], [12], ...]
sequences = tokenizer.texts_to_sequences(transactions_list)

print("Transactions converted to integer sequences.")
print("\nOriginal basket:")
print(transactions_list[0])
print("\nConverted sequence:")
print(sequences[0])

Transactions converted to integer sequences.

Original basket:
['white hanging heart t-light holder', 'white metal lantern', 'cream cupid hearts coat hanger', 'knitted union flag hot water bottle', 'red woolly hottie white heart.', 'set 7 babushka nesting boxes', 'glass star frosted t-light holder']

Converted sequence:
[1, 444, 519, 218, 255, 323, 1270]


In [29]:
print("Generating training pairs using skip-grams...")
# This process is stochastic, so results may vary slightly
pairs = []
labels = []

# Generate skip-gram pairs for each sequence
for seq in sequences:
    # window_size defines how "far" to look for context
    # negative_samples is how many fake pairs to create for each real pair
    # These settings are good for a large dataset.
    generated_pairs, generated_labels = skipgrams(
        seq, 
        vocabulary_size=vocab_size, 
        window_size=5, 
        negative_samples=1.0,
        seed=42  # <-- THIS IS THE FIX: Provide a fixed integer seed
    )
    pairs.extend(generated_pairs)
    labels.extend(generated_labels)

print(f"Generated {len(pairs)} training pairs.")

# Unzip the pairs into two separate arrays for the model
target_items = np.array([pair[0] for pair in pairs])
context_items = np.array([pair[1] for pair in pairs])
labels = np.array(labels)

print("\nExample of a training pair:")
print(f"Target: {target_items[0]}, Context: {context_items[0]}, Label: {labels[0]}")

Generating training pairs using skip-grams...
Generated 9469196 training pairs.

Example of a training pair:
Target: 255, Context: 3738, Label: 0


In [31]:
print("Building the Item2Vec neural network...")

# This is the "hyperparameter" for our model.
# It's the length of the vector for each product. 50 is a good default.
EMBEDDING_DIM = 100

# --- Define the Inputs ---
# We need two inputs, one for the target item and one for the context item
input_target = Input(shape=(1,), name="target_input")
input_context = Input(shape=(1,), name="context_input")

# --- Define the Shared Embedding Layer ---
# This layer will learn the 50-dim vector for all `vocab_size` products
embedding_layer = Embedding(
    input_dim=vocab_size, 
    output_dim=EMBEDDING_DIM, 
    name="product_embedding"
)

# --- Look up the embeddings for our inputs ---
# The target item's vector
target_embedding = embedding_layer(input_target)
# The context item's vector
context_embedding = embedding_layer(input_context)

# --- Calculate Similarity ---
# We use a Dot product to see how similar the two vectors are
# The closer the vectors, the higher the dot product
similarity = Dot(axes=2, name="similarity_dot_product")([target_embedding, context_embedding])

# Reshape the output from (batch_size, 1, 1) to (batch_size, 1)
output = Reshape((1,), name="output_reshape")(similarity)

# --- Define the Model ---
# We group the inputs and output into a final model
model = Model(inputs=[input_target, input_context], outputs=output)

# --- Compile the Model ---
# We use 'binary_crossentropy' because our labels are 0 or 1
# We add 'sigmoid' activation implicitly via the loss function
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Print a summary of our complex model
model.summary()


Building the Item2Vec neural network...


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ target_input (InputLayer)     │ (None, 1)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ context_input (InputLayer)    │ (None, 1)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ product_embedding (Embedding) │ (None, 1, 100)            │         400,000 │ target_input[0][0],        │
│                               │                           │                 │ context_input[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ similarity_dot_product (Dot)  │ (None, 1, 1)              │               0 │ product_embedding[0][0],   │
│                               │                           │                 │ product_embedding[1][0]    │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ output_reshape (Reshape)      │ (None, 1)                 │               0 │ similarity_dot_product[0]… │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 400,000 (1.53 MB)

 Trainable params: 400,000 (1.53 MB)

 Non-trainable params: 0 (0.00 B)

In [33]:
print("Training the model...")
# We'll use 20% of our data for validation to see how well it's learning
# A batch_size of 128 is efficient
history = model.fit(
    [target_items, context_items],
    labels,
    epochs=20,  # Increased from 5. This is a good starting point for the full dataset.
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

print("Model training complete.")

Training the model...
Epoch 1/20
59183/59183 ━━━━━━━━━━━━━━━━━━━━ 380s 6ms/step - accuracy: 0.7783 - loss: 0.7298 - val_accuracy: 0.7720 - val_loss: 1.0200
Epoch 2/20
59183/59183 ━━━━━━━━━━━━━━━━━━━━ 354s 6ms/step - accuracy: 0.8193 - loss: 0.6315 - val_accuracy: 0.7677 - val_loss: 1.1291
Epoch 3/20
59183/59183 ━━━━━━━━━━━━━━━━━━━━ 272s 5ms/step - accuracy: 0.8223 - loss: 0.6468 - val_accuracy: 0.7656 - val_loss: 1.2111
Epoch 4/20
59183/59183 ━━━━━━━━━━━━━━━━━━━━ 645s 11ms/step - accuracy: 0.8240 - loss: 0.6573 - val_accuracy: 0.7645 - val_loss: 1.2626
Epoch 5/20
59183/59183 ━━━━━━━━━━━━━━━━━━━━ 958s 5ms/step - accuracy: 0.8252 - loss: 0.6670 - val_accuracy: 0.7632 - val_loss: 1.2967
Epoch 6/20
59183/59183 ━━━━━━━━━━━━━━━━━━━━ 268s 5ms/step - accuracy: 0.8258 - loss: 0.6710 - val_accuracy: 0.7621 - val_loss: 1.3305
Epoch 7/20
59183/59183 ━━━━━━━━━━━━━━━━━━━━ 330s 6ms/step - accuracy: 0.8262 - loss: 0.6771 - val_accuracy: 0.7632 - val_loss: 1.3598
Epoch 8/20
59183/59183 ━━━━━━━━━━━━━━━━

In [35]:
print("Extracting learned product embeddings...")

# Get the weights from the 'product_embedding' layer
product_embeddings = model.get_layer("product_embedding").get_weights()[0]

print(f"Shape of embeddings: {product_embeddings.shape}")
print(f"(Should be: (vocab_size, EMBBEDDING_DIM))")

# We can find out which product is which using our tokenizer
# (Remember, index 0 is padding, so product '5' is at index 5)
reverse_product_index = {v: k for k, v in product_index.items()}

Extracting learned product embeddings...
Shape of embeddings: (4000, 100)
(Should be: (vocab_size, EMBBEDDING_DIM))


In [37]:
def format_for_ergo(item_name_str):
    """
    Formats the product description to be a valid ErgoAI atom.
    ErgoAI atoms cannot start with a digit, so prefix with 'n' if needed.
    """
    formatted = re.sub(r'[^a-z0-9_ ]', '', item_name_str)
    formatted = re.sub(r'\s+', '_', formatted)
    if formatted and formatted[0].isdigit():
        formatted = 'n' + formatted
    return formatted

# Test the function
test_name = "hand warmer union jack"
print(f"'{test_name}' becomes: '{format_for_ergo(test_name)}'")

'hand warmer union jack' becomes: 'hand_warmer_union_jack'


In [39]:
print(f"Calculating similarity and saving rules to {OUTPUT_FILE}...")

# Calculate the cosine similarity matrix for all embeddings
# This matrix shows the similarity score between every product
similarity_matrix = cosine_similarity(product_embeddings)

TOP_N = 5 # Find the top 5 most similar items for each item
rules_found = 0

with open(OUTPUT_FILE, 'w') as f:
    f.write("// --- Auto-generated by Neural Network Predictor ---\n\n")
    
    # Loop through every product in our vocabulary
    # (Skip index 0, which is the reserved padding token)
    for product_id in range(1, vocab_size):
        
        # Get the name of our target product
        product_a_name = reverse_product_index.get(product_id)
        if not product_a_name:
            continue
            
        # Get the similarity scores for this product against all others
        similarity_scores = similarity_matrix[product_id]
        
        # Get the indices of the top N most similar items
        # We use `-(TOP_N + 1)` because the most similar item (score 1.0)
        # will always be the product itself.
        top_indices = similarity_scores.argsort()[-(TOP_N + 1):-1][::-1]
        
        # Format for ErgoAI
        item_a = format_for_ergo(product_a_name)
        if not item_a:
            continue

        # Write a fact for each of the top N similar items
        for similar_id in top_indices:
            product_b_name = reverse_product_index.get(similar_id)
            if not product_b_name:
                continue
                
            item_b = format_for_ergo(product_b_name)
            if not item_b:
                continue
                
            # Write the fact
            fact = f"frequently_bought_together({item_a}, {item_b}).\n"
            f.write(fact)
            rules_found += 1

print(f"Successfully saved {rules_found} high-confidence association rules.")
print("\n--- NEURAL NETWORK PROCESS COMPLETE ---")
print(f"Your ErgoAI facts file is ready: {OUTPUT_FILE}")

Calculating similarity and saving rules to ergo_facts_neural_network.txt...
Successfully saved 19995 high-confidence association rules.

--- NEURAL NETWORK PROCESS COMPLETE ---
Your ErgoAI facts file is ready: ergo_facts_neural_network.txt
